* The final Chapter 8 notebook shifts from named architectures to architecture design spaces.
* AnyNet and RegNet-style thinking ask us to describe families of CNNs with stage rules, widths, depths, groups, and bottleneck ratios rather than hand-writing every layer one at a time.

# How to use this notebook

* Run the notebook from top to bottom in a clean kernel.

* The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs.

* Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

# You are done when you can

- describe a CNN as stem, stages, and head
- build a network from a compact architecture specification
- explain why design spaces are useful for architecture search
- generate RegNet-style stage widths from a simple rule
- debug grouped convolution divisibility constraints

In [4]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current

# 8.8.0 The Problem This Notebook Solves

After AlexNet, VGG, NiN, GoogLeNet, ResNet, ResNeXt, and DenseNet, a pattern should be visible:

```text
modern CNNs are built from repeatable blocks arranged into stages
```

Architecture design then becomes a search over structured choices:

- how many stages?
- how deep is each stage?
- how wide is each stage?
- where does downsampling happen?
- are convolutions dense or grouped?
- does the block use residual connections?

AnyNet describes networks from broad stage choices. RegNet narrows the design space with simple rules for stage widths and depths.

# 8.8.1 Stem, Stages, and Head

A practical CNN can often be described as:

```text
stem: early image-to-feature conversion
stages: repeated blocks at progressively lower spatial resolutions
head: global pooling and classifier
```

This vocabulary makes architectures easier to compare. Instead of memorizing a long list of layers, you can ask what each stage is doing.

In [5]:
def conv_bn_relu(in_channels, out_channels, stride=1, groups=1):

    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, groups=groups, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
    )

stem = conv_bn_relu(1, 8, stride=2)
X = torch.randn(2, 1, 64, 64)
Y = stem(X)

# Layer 0 = output size of floor(64 + 2*1 - 3)/2 + 1 = floor(32.5) = 32 for shape of (batch, out_channels, height, width) or (2, 8, 32, 32)
# Layer 1 BatchNorm2d normalizes each channel over batch, height, width; updates running_mean/running_var during training; has learnable gamma/beta
# Layer 2 ReLU

print("stem output:", shape(Y))
assert shape(Y) == (2, 8, 32, 32)

stem output: (2, 8, 32, 32)


# 8.8.2 Build AnyNet From a Stage Specification

The architecture specification below is a list of stages. Each stage says:

```text
(depth, output_channels, first_block_stride)
```

Depth means how many blocks the stage repeats. The first block may downsample; later blocks usually keep spatial size.

In [11]:
def make_stage(in_channels, depth, out_channels, first_stride):

    layers = []
    current = in_channels

    for i in range(depth):
        stride = first_stride if i == 0 else 1                                   # Except the 1st stride, all other stride=1
        layers.append(conv_bn_relu(current, out_channels, stride=stride))
        current = out_channels

    return nn.Sequential(*layers), current

class TinyAnyNet(nn.Module):

    def __init__(self, arch, num_classes=10):
        super().__init__()
        self.stem = conv_bn_relu(1, 8, stride=2)
        stages = []
        current = 8

        for depth, out_channels, stride in arch:
            stage, current = make_stage(current, depth, out_channels, stride)    # Starting in_channels is 8
            stages.append(stage)

        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)),
                                  nn.Flatten(),
                                  nn.Linear(current, num_classes))               # Produce the classification head after all convolutions

    def forward(self, X):
        return self.head(self.stages(self.stem(X)))

anynet = TinyAnyNet(arch=[(1, 16, 2), (2, 32, 2), (1, 64, 2)])
logits = anynet(torch.randn(2, 1, 64, 64))

# nn.Sequential(make_stage(8, 1, 16, 2), make_stage(16, 2, 32, 2), make_stage(32, 1, 64, 2),
              # nn.AdaptiveAvgPool2d((1, 1), nn.Flatten(), nn.Linear(current, num_classes))

# Input
# (2, 1, 64, 64)
#        ↓
# Stem: 1 → 8, stride 2
# (2, 8, 32, 32)
#        ↓
# Stage 1: 8 → 16, stride 2
# (2, 16, 16, 16)
#        ↓
# Stage 2:
# 16 → 32, stride 2
# 32 → 32, stride 1
# (2, 32, 8, 8)
#        ↓
# Stage 3: 32 → 64, stride 2
# (2, 64, 4, 4)
#        ↓
# AdaptiveAvgPool
# (2, 64, 1, 1)
#        ↓
# Flatten
# (2, 64)
#        ↓
# Linear(64, 10)
# (2, 10)

print("logits:", shape(logits))
assert shape(logits) == (2, 10)

AttributeError: 'Tensor' object has no attribute 'stagestatus'

# 8.8.3 Compare Design Choices With Parameter Counts

A design space is useful because it lets you compare many related models systematically. The cell compares two specifications:

- shallow and wider
- deeper and narrower

Parameter count is not the only metric.

Runtime, memory access, accuracy, and hardware fit also matter.

But parameter counting is a concrete first inspection tool.

In [7]:
candidate_arches = {
    "shallow_wide": [(1, 24, 2), (1, 48, 2), (1, 96, 2)],                        # fewer layers, more channels
    "deeper_narrow": [(2, 16, 2), (2, 32, 2), (2, 64, 2)],                       # more layers, fewer channels
}

for name, arch in candidate_arches.items():
    model = TinyAnyNet(arch)
    logits = model(torch.randn(2, 1, 64, 64))
    print(name, "params", count_parameters(model), "logits", shape(logits))
    assert shape(logits) == (2, 10)                                              # Both produce shape of (2, 10), however shallow_wide has fewer parameters since it has less steps (depth)

shallow_wide params 54962 logits (2, 10)
deeper_narrow params 73762 logits (2, 10)


# 8.8.4 A RegNet-Style Width Rule

RegNet-style design narrows the search space by generating widths with a simple rule, then grouping repeated equal widths into stages.

This simplified version uses:

```text
raw width at block i = w0 + wa * i
quantize raw widths to multiples of q
merge consecutive blocks with the same quantized width into stages
```

The real design space includes more details. This drill captures the key abstraction: generate architecture structure from few parameters.

In [8]:
def quantize(width, q):
    return int(round(width / q) * q)

def regnet_like_stages(depth, w0, wa, q):
    widths = [max(q, quantize(w0 + wa * i, q)) for i in range(depth)]
    stages = []

    for width in widths:
        if stages and stages[-1][1] == width:
            stages[-1] = (stages[-1][0] + 1, width)
        else:
            stages.append((1, width))

    return widths, stages

widths, stages = regnet_like_stages(depth=8, w0=16, wa=7, q=8)
print("block widths:", widths)
print("merged stages:", stages)

assert len(widths) == 8
assert sum(depth for depth, width in stages) == 8
assert all(width % 8 == 0 for width in widths)

block widths: [16, 24, 32, 40, 48, 48, 56, 64]
merged stages: [(1, 16), (1, 24), (1, 32), (1, 40), (2, 48), (1, 56), (1, 64)]


# 8.8.5 Convert Generated Stages Into a Network

The generated stage list lacks stride information, so this cell adds a simple policy:

```text
downsample at the first block of every generated stage
```

This policy is not universal. The point is to separate design-space generation from model construction.

In [9]:
generated_arch = [(depth, width, 2) for depth, width in stages[:3]]              # Prepares for (depth, out_channels, stride) shape for the first 3 pairs of depth & width with stride=2
                                                                                 # Generates [(?, 16, 2), (?, 24, 2), (?, 32, 2)]
print("generated arch:", generated_arch)
model = TinyAnyNet(generated_arch)
X = torch.randn(2, 1, 64, 64)
rows = []
current = model.stem(X)                                                          # Hike channels to 8 and cut spatial dimensions by half, or (2, 8, 32, 32)
rows.append(("stem", shape(current)))

for i, stage in enumerate(model.stages):
    current = stage(current)
    rows.append((f"stage_{i}", shape(current)))
logits = model.head(current)

print(rows)
print("logits:", shape(logits))

# Input
# (2, 1, 64, 64)
#        │
#        │ stride 2
#        ▼
# Stem
# (2, 8, 32, 32)
#        │
#        │ stride 2
#        ▼
# Stage 0
# (2, 16, 16, 16)
#        │
#        │ stride 2
#        ▼
# Stage 1
# (2, 24, 8, 8)
#        │
#        │ stride 2
#        ▼
# Stage 2
# (2, 32, 4, 4)
#        │
#        │ AdaptiveAvgPool
#        ▼
# (2, 32, 1, 1)
#        │
#        │ Flatten
#        ▼
# (2, 32)
#        │
#        │ Linear 32 → 10
#        ▼
# Logits
# (2, 10)

assert shape(logits)
assert rows[-1][1][1] == generated_arch[-1][1]

[('stem', (2, 8, 32, 32)), ('stage_0', (2, 16, 16, 16)), ('stage_1', (2, 24, 8, 8)), ('stage_2', (2, 32, 4, 4))]
logits: (2, 10)


# 8.8.6 Break It Deliberately: Grouped Convolution Divisibility

Grouped convolution imposes a hard channel contract:

```text
input channels must be divisible by groups
output channels must be divisible by groups
```

Design-space search must respect these arithmetic constraints. Otherwise the model cannot even be constructed.

In [12]:
try:
    nn.Conv2d(10, 16, kernel_size=3, padding=1, groups=4)                        # 10 / 4 is not an integer, so 10 input channels cannot be split into 4 groups
except ValueError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected grouped convolution construction to fail")

ValueError
in_channels must be divisible by groups


# 8.8.7 What This Chapter Does Not Do

This chapter does not run architecture search, train RegNet variants, benchmark GPUs, compare ImageNet accuracy, tune regularization, or report production metrics.

Those tasks require real datasets, controlled training budgets, and hardware-aware measurement. The chapter goal is narrower and more foundational:

- read modern CNNs as combinations of reusable blocks
- trace shape and channel contracts
- understand why bottlenecks, branches, residuals, dense concatenation, normalization, and design spaces exist
- recognize common architecture failures before large training runs hide them

# 8.8 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. What are stem, stages, and head in a CNN architecture?
> The stem converts the input image into early features. Stages contain repeated blocks that progressively change spatial resolution and channel width. The head uses global pooling and a classifier to produce the final predictions

2. Why is a design space different from one fixed model?
> A design space describes a family of related models using choices such as stage depth, width, stride, and block type, rather than specifying one fixed architecture

3. What does stage depth control?
> Stage depth controls how many blocks are repeated within a stage. Greater depth generally increases the number of parameters and computation

4. Why do RegNet-style rules generate widths before constructing layers?
> RegNet-style rules generate and quantize block widths first, then merge equal consecutive widths into stages. This lets the architecture be generated systematically from a few design parameters instead of specifying every layer manually

5. Why must grouped convolution constraints be checked during architecture design?
> Grouped convolution requires both input and output channels to be divisible by the number of groups. These constraints must be checked during architecture design because an invalid configuration cannot be constructed